# Camada Silver — Limpeza, Padronização e Controle de Alterações
### Projeto: Antecipação de Alertas Críticos em Frotas de Mineração

Parto das tabelas que já gravei na Bronze (`apontamentos`, `telemetria`,
`alarmes_regras`) e produzo as tabelas da Silver: dados tipados corretamente, sem
duplicatas, sem "nulos disfarçados" de string, com as inconsistências temporais
tratadas — e com uma tabela de Controle de Alterações (ANTES/DEPOIS) documentando
cada decisão de limpeza que eu tomei, como o próprio Estudo Guiado pede.

## 0. Setup e carga da Bronze

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Lendo as 3 tabelas que gravei na camada Bronze
df_apontamentos = spark.table("`estudo-guiado`.vale_bronze.apontamentos")
df_telemetria   = spark.table("`estudo-guiado`.vale_bronze.telemetria")
df_alarmes      = spark.table("`estudo-guiado`.vale_bronze.alarmes_regras")

# Lista onde vou guardando o Controle de Alterações (ANTES/DEPOIS) pedido no Estudo
# Guiado. Cada vez que eu tomo uma decisão de limpeza, registro aqui — no final vira
# uma tabela auditável, provando que cada mudança foi consciente e justificada.
controle_alteracoes = []

def registrar(tabela, campo, problema, qtd_registros, tratamento, justificativa):
    """Adiciona uma linha ao meu log de Controle de Alterações."""
    controle_alteracoes.append({
        "Tabela": tabela,
        "Campo": campo,
        "Problema_Identificado": problema,
        "Qtd_Registros": int(qtd_registros),
        "Tratamento_Aplicado": tratamento,
        "Justificativa": justificativa,
    })

## 1. Diagnóstico inicial (antes de mexer em qualquer coisa)
Shape, tipos, nulos e duplicatas — essa é a minha linha de base do "ANTES" no
controle de alterações.

In [0]:
def diagnostico(df, nome):
    print(f"\n=== Diagnóstico: {nome} ===")
    print(f"Linhas: {df.count():,} | Colunas: {len(df.columns)}")
    print("\nTipos de dados:")
    df.printSchema()
    print("\nContagem de nulos reais por coluna:")
    display(
        df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
    )

diagnostico(df_apontamentos, "Apontamentos (Bronze)")
diagnostico(df_telemetria, "Telemetria (Bronze)")
diagnostico(df_alarmes, "Regras de Alarmes (Bronze)")


=== Diagnóstico: Apontamentos (Bronze) ===
Linhas: 377,907 | Colunas: 7

Tipos de dados:
root
 |-- Id: long (nullable = true)
 |-- Inicio: string (nullable = true)
 |-- Fim: string (nullable = true)
 |-- Tag: string (nullable = true)
 |-- Frota: string (nullable = true)
 |-- Tipo: string (nullable = true)
 |-- Classe: string (nullable = true)


Contagem de nulos reais por coluna:


Id,Inicio,Fim,Tag,Frota,Tipo,Classe
0,0,0,0,0,0,0



=== Diagnóstico: Telemetria (Bronze) ===
Linhas: 37,164,054 | Colunas: 18

Tipos de dados:
root
 |-- Id_Eventos_Telemetria: long (nullable = true)
 |-- Data_Evento: timestamp_ntz (nullable = true)
 |-- Inicio_Turno: string (nullable = true)
 |-- Fim_Turno: string (nullable = true)
 |-- Dia: long (nullable = true)
 |-- Localidade: string (nullable = true)
 |-- TAG: string (nullable = true)
 |-- Tag_Frota: string (nullable = true)
 |-- Tipo: string (nullable = true)
 |-- Nome_Operador_Anon: string (nullable = true)
 |-- Matricula_Operador_Hash: string (nullable = true)
 |-- Id_Alarme: long (nullable = true)
 |-- Alarme: string (nullable = true)
 |-- Id_Criticidade: long (nullable = true)
 |-- Criticidade: string (nullable = true)
 |-- Valor: string (nullable = true)
 |-- Classe: string (nullable = true)
 |-- Is_Dont_Go: byte (nullable = true)


Contagem de nulos reais por coluna:


Id_Eventos_Telemetria,Data_Evento,Inicio_Turno,Fim_Turno,Dia,Localidade,TAG,Tag_Frota,Tipo,Nome_Operador_Anon,Matricula_Operador_Hash,Id_Alarme,Alarme,Id_Criticidade,Criticidade,Valor,Classe,Is_Dont_Go
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



=== Diagnóstico: Regras de Alarmes (Bronze) ===
Linhas: 152 | Colunas: 6

Tipos de dados:
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)


Contagem de nulos reais por coluna:


_c0,_c1,_c2,_c3,_c4,_c5
0,0,0,0,0,0


### 1.1 Nulos "disfarçados" de string
O diagnóstico acima usa `isNull()`, que só pega nulo real. Só que percebi que a
coluna `Classe` não tem nulo real — tem a string literal `"null"` (texto), por isso
o diagnóstico mostrou 0 nulos em tudo, o que é enganoso. Antes de decidir o que
tratar, faço uma varredura genérica em todas as colunas de texto pra não deixar
esse padrão passar despercebido em outra coluna (Frota, Alarme, Criticidade,
Localidade, etc.).

In [0]:
def contar_nulos_textuais(df, nome):
    """Conta nulo real + variações textuais que representam ausência de dado
    ('null', 'none', 'nan', 'n/a', string vazia) — mesmo em colunas sem NULL de verdade."""
    colunas_string = [c for c, t in df.dtypes if t == "string"]
    condicoes = [
        F.count(F.when(
            F.col(c).isNull() | F.lower(F.trim(F.col(c))).isin("null", "none", "nan", "n/a", ""),
            c
        )).alias(c)
        for c in colunas_string
    ]
    print(f"\n=== Nulos reais + textuais disfarçados: {nome} ===")
    display(df.select(condicoes))

contar_nulos_textuais(df_apontamentos, "Apontamentos (Bronze)")
contar_nulos_textuais(df_telemetria, "Telemetria (Bronze)")

# Antes de seguir, olho o resultado dessa célula. Se aparecer alguma coluna além de
# Classe/Valor com contagem relevante, ela também precisa entrar no tratamento genérico
# lá da seção 6 (o código já é genérico e cobre isso automaticamente).


=== Nulos reais + textuais disfarçados: Apontamentos (Bronze) ===


Inicio,Fim,Tag,Frota,Tipo,Classe
0,0,0,0,0,0



=== Nulos reais + textuais disfarçados: Telemetria (Bronze) ===


Inicio_Turno,Fim_Turno,Localidade,TAG,Tag_Frota,Tipo,Nome_Operador_Anon,Matricula_Operador_Hash,Alarme,Criticidade,Valor,Classe
0,0,0,0,0,0,0,0,0,0,237443,36104611


## 2. Tipagem — `df_apontamentos`
Cast dinâmico: datas viram timestamp, o resto mantém o tipo que já veio certo da Bronze.

In [0]:
colunas_ap_transformadas = []
for coluna, tipo in df_apontamentos.dtypes:
    if coluna in ["Inicio", "Fim"]:
        expressao = F.col(coluna).cast("timestamp")
    elif tipo == "string":
        expressao = F.col(coluna).cast("string")
    else:
        expressao = F.col(coluna)
    colunas_ap_transformadas.append(expressao.alias(coluna))

df_apontamentos_silver = df_apontamentos.select(colunas_ap_transformadas)
print("Schema de Apontamentos após tipagem:")
df_apontamentos_silver.printSchema()

Schema de Apontamentos após tipagem:
root
 |-- Id: long (nullable = true)
 |-- Inicio: timestamp (nullable = true)
 |-- Fim: timestamp (nullable = true)
 |-- Tag: string (nullable = true)
 |-- Frota: string (nullable = true)
 |-- Tipo: string (nullable = true)
 |-- Classe: string (nullable = true)



## 3. Tipagem — `df_telemetria`
Mesma lógica, mais o tratamento de `Valor` (vírgula decimal + string "NULL") e `Dia`
como inteiro (a coluna veio como long, não precisa disso).

In [0]:
colunas_tl_transformadas = []
for coluna, tipo in df_telemetria.dtypes:
    if coluna.lower().endswith("turno") or coluna.lower() == "data_evento":
        expressao = F.col(coluna).cast("timestamp")
    elif coluna.lower() == "valor":
        expressao = F.when(
            (F.upper(F.col(coluna)) == "NULL") | (F.col(coluna) == ""),
            F.lit(None)
        ).otherwise(
            F.regexp_replace(F.col(coluna), ",", ".").cast("double")
        )
    elif coluna.lower() == "dia":
        expressao = F.col(coluna).cast("integer")
    elif tipo == "string":
        expressao = F.col(coluna).cast("string")
    else:
        expressao = F.col(coluna)
    colunas_tl_transformadas.append(expressao.alias(coluna))

df_telemetria_silver = df_telemetria.select(colunas_tl_transformadas)
print("Schema de Telemetria após tipagem:")
df_telemetria_silver.printSchema()

Schema de Telemetria após tipagem:
root
 |-- Id_Eventos_Telemetria: long (nullable = true)
 |-- Data_Evento: timestamp (nullable = true)
 |-- Inicio_Turno: timestamp (nullable = true)
 |-- Fim_Turno: timestamp (nullable = true)
 |-- Dia: integer (nullable = true)
 |-- Localidade: string (nullable = true)
 |-- TAG: string (nullable = true)
 |-- Tag_Frota: string (nullable = true)
 |-- Tipo: string (nullable = true)
 |-- Nome_Operador_Anon: string (nullable = true)
 |-- Matricula_Operador_Hash: string (nullable = true)
 |-- Id_Alarme: long (nullable = true)
 |-- Alarme: string (nullable = true)
 |-- Id_Criticidade: long (nullable = true)
 |-- Criticidade: string (nullable = true)
 |-- Valor: double (nullable = true)
 |-- Classe: string (nullable = true)
 |-- Is_Dont_Go: byte (nullable = true)



## 4. Regras de Alarmes — limpeza do catálogo
A Bronze leu o Excel sem reconhecer o cabeçalho corretamente, então as colunas vêm
como `_c0..._c5` e a primeira linha de dados é, na verdade, o cabeçalho original
(TIPO, EVENTO, SITUACAO, QTD, TEMPO, NIVEL). Trato isso aqui, filtrando essa linha
de forma case-insensitive (pra não escapar nenhuma variação de capitalização) e já
renomeando as colunas pro nome certo. O ideal a médio prazo seria corrigir a leitura
na própria Bronze (opção de header/dataAddress do conector de Excel), mas por ora
resolvo aqui pra não perder a regra de negócio.

In [0]:
df_regras_filtrado = df_alarmes.filter(
    (F.lower(F.col("_c0")) != "tipo") &
    (F.lower(F.col("_c3")) != "qtd") &
    (F.lower(F.col("_c4")) != "tempo")
)

qtd_linhas_cabecalho = df_alarmes.count() - df_regras_filtrado.count()

df_regras_silver = df_regras_filtrado.select(
    F.trim(F.lower(F.col("_c0"))).cast("string").alias("tipo"),
    F.trim(F.lower(F.col("_c1"))).cast("string").alias("evento"),
    F.trim(F.lower(F.col("_c2"))).cast("string").alias("situacao"),
    F.expr("try_cast(_c3 as int)").alias("qtd"),
    F.expr("try_cast(_c4 as int)").alias("tempo"),
    F.trim(F.lower(F.col("_c5"))).cast("string").alias("nivel"),
)

registrar(
    tabela="df_regras_silver",
    campo="_c0..c5 (cabeçalho embutido)",
    problema="Linha de cabeçalho textual (TIPO/QTD/TEMPO) veio junto com os dados por falha de leitura na Bronze",
    qtd_registros=qtd_linhas_cabecalho,
    tratamento="Linha removida + colunas renomeadas para tipo/evento/situacao/qtd/tempo/nivel",
    justificativa="Conector de Excel na Bronze não reconheceu o header corretamente; tratado aqui para não perder a regra de negócio, mas recomendo corrigir a leitura na origem quando possível.",
)

print(f"Linhas de cabeçalho removidas do catálogo de regras: {qtd_linhas_cabecalho}")
df_regras_silver.printSchema()

Linhas de cabeçalho removidas do catálogo de regras: 1
root
 |-- tipo: string (nullable = true)
 |-- evento: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- qtd: integer (nullable = true)
 |-- tempo: integer (nullable = true)
 |-- nivel: string (nullable = true)



## 5. Padronização de texto (trim + lowercase)
Preciso colocar tudo em minúsculo e sem espaço nas pontas pra não contar a mesma
categoria como grupos diferentes. Mas tomo cuidado com uma coisa: `Matricula_Operador_Hash`
não pode entrar nessa padronização — hash costuma ser case-sensitive, e colocar tudo
em minúsculo pode fazer dois operadores diferentes colidirem no mesmo valor,
corrompendo silenciosamente qualquer análise de comportamento do operador (uma das
perguntas de negócio que quero responder). Por isso, excluo identificadores e hashes
dessa padronização e só aplico lower/trim nas colunas que são categoria de verdade.

In [0]:
# Colunas que são identificador/hash/chave e não podem ter a caixa alterada
colunas_preservar_caixa = {
    "id", "tag", "tag_frota",
    "id_eventos_telemetria", "id_alarme", "id_criticidade",
    "matricula_operador_hash", "nome_operador_anon",
}

def padronizar_texto(df):
    colunas_transformadas = [
        F.trim(F.lower(F.col(c))).alias(c)
        if (t == "string" and c.lower() not in colunas_preservar_caixa)
        else F.col(c)
        for c, t in df.dtypes
    ]
    return df.select(colunas_transformadas)

df_apontamentos_silver = padronizar_texto(df_apontamentos_silver)
df_telemetria_silver   = padronizar_texto(df_telemetria_silver)
df_regras_silver       = padronizar_texto(df_regras_silver)

print("Padronização de caixa (lowercase) e trim concluída — preservando Tag/hash/IDs.")

registrar(
    tabela="apontamentos / telemetria / regras",
    campo="colunas de texto categóricas (Frota, Tipo, Classe, Alarme, Criticidade, Localidade, tipo/evento/situacao/nivel)",
    problema="Capitalização e espaços em branco inconsistentes entre categorias",
    qtd_registros=df_telemetria_silver.count(),
    tratamento="trim() + lower() aplicado, exceto em Tag/TAG, hash do operador e colunas de Id",
    justificativa="Evita que a mesma categoria seja contada como grupos distintos, sem correr o risco de corromper identificadores e hashes de anonimização, que são case-sensitive por natureza.",
)

Padronização de caixa (lowercase) e trim concluída — preservando Tag/hash/IDs.


## 6. Tratamento genérico de nulos / "null" textual
Em vez de tratar coluna por coluna manualmente, escrevi uma função genérica: qualquer
coluna categórica com nulo real ou "null" textual recebe o rótulo "não informado".
Assim, se aparecer mais alguma coluna com esse problema (além de Classe, que foi
onde eu vi isso pela primeira vez), já está coberta sem precisar escrever código novo.

In [0]:
def tratar_nulos_textuais_categoricos(df, colunas_categoricas, rotulo="não informado"):
    """Substitui nulo real + variações textuais de nulo por um rótulo explícito,
    em vez de descartar a linha inteira — a ausência da categoria é informação válida."""
    for coluna in colunas_categoricas:
        if coluna not in df.columns:
            continue
        qtd_afetada = df.filter(
            F.col(coluna).isNull() | F.lower(F.trim(F.col(coluna))).isin("null", "none", "nan", "n/a", "")
        ).count()
        if qtd_afetada > 0:
            df = df.withColumn(
                coluna,
                F.when(
                    F.col(coluna).isNull() | F.lower(F.trim(F.col(coluna))).isin("null", "none", "nan", "n/a", ""),
                    F.lit(rotulo)
                ).otherwise(F.col(coluna))
            )
            registrar(
                tabela="apontamentos/telemetria",
                campo=coluna,
                problema="Nulo real ou string 'null'/'none'/'n/a' representando ausência de dado",
                qtd_registros=qtd_afetada,
                tratamento=f"Preenchido com '{rotulo}'",
                justificativa="Ausência da categoria é informação válida (equipamento/evento sem essa classificação), não deve ser descartada nem confundida com dado sujo.",
            )
    return df

colunas_categoricas_apontamentos = ["frota", "tipo", "classe"]
colunas_categoricas_telemetria = ["localidade", "tag_frota", "tipo", "alarme", "criticidade", "classe"]

df_apontamentos_silver = tratar_nulos_textuais_categoricos(df_apontamentos_silver, colunas_categoricas_apontamentos)
df_telemetria_silver   = tratar_nulos_textuais_categoricos(df_telemetria_silver, colunas_categoricas_telemetria)

## 7. Coluna `Valor` (Telemetria) — nulos numéricos
Pensei em preencher o `Valor` nulo com 0.0, mas percebi que isso seria arriscado:
zero pode ser uma leitura real do sensor, dependendo do tipo de alarme. Preencher com
0 misturaria "sem leitura" com "leitura igual a zero", o que pode confundir o modelo
depois. Preferi criar uma flag e deixar a decisão de imputação pra etapa de
modelagem, quando dá pra olhar isso com mais contexto (por tipo de alarme).

In [0]:
qtd_valor_nulo = df_telemetria_silver.filter(F.col("valor").isNull()).count()

df_telemetria_silver = df_telemetria_silver.withColumn(
    "valor_ausente", F.col("valor").isNull()
)

registrar(
    tabela="df_telemetria_silver",
    campo="valor",
    problema="Valor nulo após conversão para double (não numérico ou ausente na origem)",
    qtd_registros=qtd_valor_nulo,
    tratamento="Mantido nulo + criada a coluna flag 'valor_ausente' (não preenchido com 0.0)",
    justificativa="Zero pode ser uma leitura legítima do sensor para alguns tipos de alarme; imputar 0 cegamente confundiria 'sem leitura' com 'leitura zero'. A flag permite decidir a imputação por tipo de alarme na etapa de modelagem.",
)
print(f"Valores nulos em 'valor': {qtd_valor_nulo:,} — sinalizados em 'valor_ausente', não preenchidos.")

Valores nulos em 'valor': 237,443 — sinalizados em 'valor_ausente', não preenchidos.


## 8. Inconsistências temporais em Apontamentos
Preciso checar `Inicio > Fim`, duração de ciclo nula/negativa, e sinalizar (sem
remover) ciclos sobrepostos no mesmo equipamento.

In [0]:
df_apontamentos_silver = df_apontamentos_silver.withColumn(
    "duracao_minutos",
    (F.col("fim").cast("long") - F.col("inicio").cast("long")) / 60.0
)

qtd_antes_temporal = df_apontamentos_silver.count()
qtd_inicio_maior_fim = df_apontamentos_silver.filter(F.col("inicio") > F.col("fim")).count()
qtd_duracao_invalida = df_apontamentos_silver.filter(F.col("duracao_minutos") <= 0).count()

df_apontamentos_silver = df_apontamentos_silver.filter(
    (F.col("inicio") <= F.col("fim")) & (F.col("duracao_minutos") > 0)
)
qtd_removidos_temporal = qtd_antes_temporal - df_apontamentos_silver.count()

registrar(
    tabela="df_apontamentos_silver",
    campo="inicio, fim, duracao_minutos",
    problema="Inicio > Fim ou duração do ciclo <= 0",
    qtd_registros=qtd_removidos_temporal,
    tratamento="Linha removida",
    justificativa="Ciclo com duração nula/negativa não representa uma atividade real e distorceria qualquer feature de tempo de operação.",
)
print(f"Removidos {qtd_removidos_temporal:,} ciclos com Inicio>Fim ou duração inválida "
      f"(Inicio>Fim: {qtd_inicio_maior_fim} | duração<=0: {qtd_duracao_invalida}).")

# Sobreposição de ciclos no mesmo equipamento — só sinalizo, não removo. Overlap pode
# ser troca de operador/turno legítima ou erro real de apontamento; decidir isso exige
# olhar caso a caso na EDA, não dá pra resolver com uma regra automática cega.
janela_tag = Window.partitionBy("tag").orderBy("inicio")
df_apontamentos_silver = df_apontamentos_silver.withColumn(
    "fim_ciclo_anterior", F.lag("fim").over(janela_tag)
).withColumn(
    "flag_overlap_ciclo", F.when(F.col("inicio") < F.col("fim_ciclo_anterior"), True).otherwise(False)
).drop("fim_ciclo_anterior")

qtd_overlap = df_apontamentos_silver.filter(F.col("flag_overlap_ciclo")).count()
print(f"Ciclos sinalizados com sobreposição (flag_overlap_ciclo=True): {qtd_overlap:,}")

registrar(
    tabela="df_apontamentos_silver",
    campo="flag_overlap_ciclo",
    problema="Ciclo com Inicio anterior ao Fim do ciclo anterior do mesmo equipamento",
    qtd_registros=qtd_overlap,
    tratamento="Não removido — apenas sinalizado com flag_overlap_ciclo=True",
    justificativa="Sobreposição pode ser troca de operador/turno legítima ou erro de apontamento; decisão de remover exige análise caso a caso, não uma regra cega.",
)

Removidos 0 ciclos com Inicio>Fim ou duração inválida (Inicio>Fim: 0 | duração<=0: 0).
Ciclos sinalizados com sobreposição (flag_overlap_ciclo=True): 351


## 9. Outliers de duração de ciclo
Uso o critério de IQR (Interquartile Range) sobre `duracao_minutos`. Só sinalizo —
um ciclo "Parado" pode legitimamente durar muitas horas, então não faz sentido
remover automaticamente.

In [0]:
q1, q3 = df_apontamentos_silver.approxQuantile("duracao_minutos", [0.25, 0.75], 0.01)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

df_apontamentos_silver = df_apontamentos_silver.withColumn(
    "flag_outlier_duracao", F.col("duracao_minutos") > F.lit(limite_superior)
)

qtd_outliers = df_apontamentos_silver.filter(F.col("flag_outlier_duracao")).count()
print(f"Q1={q1:.1f} min | Q3={q3:.1f} min | limite superior IQR={limite_superior:.1f} min")
print(f"Ciclos sinalizados como outlier de duração: {qtd_outliers:,}")

registrar(
    tabela="df_apontamentos_silver",
    campo="duracao_minutos",
    problema=f"Duração de ciclo acima do limite IQR (> {limite_superior:.0f} min)",
    qtd_registros=qtd_outliers,
    tratamento="Não removido — sinalizado com flag_outlier_duracao=True",
    justificativa="Ciclos 'Parado' podem durar muitas horas legitimamente; remover cegamente distorceria a análise de equipamentos parados por manutenção real.",
)

Q1=6.1 min | Q3=60.0 min | limite superior IQR=140.8 min
Ciclos sinalizados como outlier de duração: 0


## 10. Deduplicação
Apontamentos por `id`, Telemetria por `id_eventos_telemetria`, Regras pela
combinação `tipo + evento + situacao`.

In [0]:
qtd_ap_antes = df_apontamentos_silver.count()
qtd_tl_antes = df_telemetria_silver.count()
qtd_rg_antes = df_regras_silver.count()

df_apontamentos_silver = df_apontamentos_silver.dropDuplicates(["id"])
df_telemetria_silver = df_telemetria_silver.dropDuplicates(["id_eventos_telemetria"])
df_regras_silver = df_regras_silver.dropDuplicates(["tipo", "evento", "situacao"])

qtd_ap_dup = qtd_ap_antes - df_apontamentos_silver.count()
qtd_tl_dup = qtd_tl_antes - df_telemetria_silver.count()
qtd_rg_dup = qtd_rg_antes - df_regras_silver.count()

registrar("df_apontamentos_silver", "id", "Registros duplicados", qtd_ap_dup,
          "dropDuplicates(['id'])", "Id é chave única do ciclo; duplicidade indica reingestão do bronze.")
registrar("df_telemetria_silver", "id_eventos_telemetria", "Registros duplicados", qtd_tl_dup,
          "dropDuplicates(['id_eventos_telemetria'])", "Chave única do evento; duplicidade vem de overlap entre arquivos mensais.")
registrar("df_regras_silver", "tipo, evento, situacao", "Regras duplicadas", qtd_rg_dup,
          "dropDuplicates(['tipo','evento','situacao'])", "Mesma combinação de regra não deve aparecer duas vezes no catálogo.")

print(f"Duplicatas removidas -> Apontamentos: {qtd_ap_dup} | Telemetria: {qtd_tl_dup} | Regras: {qtd_rg_dup}")

Duplicatas removidas -> Apontamentos: 0 | Telemetria: 0 | Regras: 0


## 11. Validação final pós-limpeza

In [0]:
def validar(df, chave, nome):
    total = df.count()
    distintos = df.select(chave).distinct().count()
    print(f"{nome}: {total:,} linhas | {distintos:,} valores distintos em '{chave}' "
          f"| duplicatas remanescentes: {total - distintos}")

validar(df_apontamentos_silver, "id", "Apontamentos (Silver)")
validar(df_telemetria_silver, "id_eventos_telemetria", "Telemetria (Silver)")

print("\n--- Pré-visualização final: Apontamentos ---")
display(df_apontamentos_silver.limit(10))
print("\n--- Pré-visualização final: Telemetria ---")
display(df_telemetria_silver.limit(10))
print("\n--- Pré-visualização final: Regras de Alarmes ---")
display(df_regras_silver.limit(10))

Apontamentos (Silver): 377,907 linhas | 377,907 valores distintos em 'id' | duplicatas remanescentes: 0
Telemetria (Silver): 37,164,054 linhas | 37,164,054 valores distintos em 'id_eventos_telemetria' | duplicatas remanescentes: 0

--- Pré-visualização final: Apontamentos ---


Id,Inicio,Fim,Tag,Frota,Tipo,Classe,duracao_minutos,flag_overlap_ciclo,flag_outlier_duracao
23132929,2025-01-01T01:00:00.000Z,2025-01-01T01:37:05.000Z,CA5927,793-d 5s,caminhao,parado,37.083333333333336,false,false
23143782,2025-01-01T05:09:18.000Z,2025-01-01T06:00:00.000Z,CA5927,793-d 5s,caminhao,operando,50.7,false,false
23148275,2025-01-01T07:00:00.000Z,2025-01-01T07:22:41.000Z,CA5927,793-d 5s,caminhao,parado,22.683333333333334,false,false
23154580,2025-01-01T08:59:05.000Z,2025-01-01T09:00:00.000Z,CA5927,793-d 5s,caminhao,parado,0.9166666666666666,false,false
23159199,2025-01-01T10:00:00.000Z,2025-01-01T10:03:36.000Z,CA5927,793-d 5s,caminhao,parado,3.6,false,false
23254524,2025-01-02T04:50:32.000Z,2025-01-02T05:00:00.000Z,CA5927,793-d 5s,caminhao,parado,9.466666666666667,false,false
23262406,2025-01-02T07:00:00.000Z,2025-01-02T07:17:04.000Z,CA5927,793-d 5s,caminhao,operando,17.066666666666666,false,false
23266397,2025-01-02T08:28:09.000Z,2025-01-02T08:30:42.000Z,CA5927,793-d 5s,caminhao,parado,2.55,false,false
23266556,2025-01-02T08:30:42.000Z,2025-01-02T09:00:00.000Z,CA5927,793-d 5s,caminhao,operando,29.3,false,false
23321148,2025-01-02T19:00:00.000Z,2025-01-02T20:00:00.000Z,CA5927,793-d 5s,caminhao,manutenção,60.0,false,false



--- Pré-visualização final: Telemetria ---


Id_Eventos_Telemetria,Data_Evento,Inicio_Turno,Fim_Turno,Dia,Localidade,TAG,Tag_Frota,Tipo,Nome_Operador_Anon,Matricula_Operador_Hash,Id_Alarme,Alarme,Id_Criticidade,Criticidade,Valor,Classe,Is_Dont_Go,valor_ausente
130322552,2025-06-01T07:20:48.850Z,2025-06-01T06:00:00.000Z,2025-06-01T18:00:00.000Z,1,itabira,CA5926,793-D 5S,caminhao,OP_055,H_39dd7dffa0cd,84609758,hoist lever sensor - inactive,3,informacional,2.0,inactive,0,false
131527474,2025-06-01T22:18:35.073Z,2025-06-01T18:00:00.000Z,2025-06-02T06:00:00.000Z,1,itabira,CA5926,793-D 5S,caminhao,OP_153,H_c3ea40faae6b,84626976,dipper,3,informacional,118.200004577637,null,0,false
132064853,2025-06-02T03:21:55.440Z,2025-06-01T18:00:00.000Z,2025-06-02T06:00:00.000Z,2,itabira,CA5926,793-D 5S,caminhao,OP_153,H_c3ea40faae6b,84626976,dipper,3,informacional,252.699996948242,null,0,false
38396366,2025-06-02T16:24:55.483Z,2025-06-02T06:00:00.000Z,2025-06-02T18:00:00.000Z,2,itabira,CA5926,793-D 5S,caminhao,OP_375,H_a6087897523d,16834354,mc - queda pressão do óleo motor <280kpa,2,não crítico,null,null,0,true
135646036,2025-06-04T09:23:14.993Z,2025-06-04T06:00:00.000Z,2025-06-04T18:00:00.000Z,4,itabira,CA5926,793-D 5S,caminhao,OP_275,H_4c1bffed2f6b,335609935,rx channel b not receiving messages,3,informacional,0.0,null,0,false
138751104,2025-06-06T00:39:08.413Z,2025-06-05T18:00:00.000Z,2025-06-06T06:00:00.000Z,6,itabira,CA5926,793-D 5S,caminhao,OP_166,H_addd678d0fb8,84626977,load,3,informacional,233.100006103516,null,0,false
138812909,2025-06-06T01:02:59.337Z,2025-06-05T18:00:00.000Z,2025-06-06T06:00:00.000Z,6,itabira,CA5926,793-D 5S,caminhao,OP_166,H_addd678d0fb8,335609934,rx channel a not receiving messages,3,informacional,0.0,null,0,false
139133666,2025-06-06T05:23:37.683Z,2025-06-05T18:00:00.000Z,2025-06-06T06:00:00.000Z,6,itabira,CA5926,793-D 5S,caminhao,OP_166,H_addd678d0fb8,84626976,dipper,3,informacional,99.0,null,0,false
139240921,2025-06-06T07:24:06.517Z,2025-06-06T06:00:00.000Z,2025-06-06T18:00:00.000Z,6,itabira,CA5926,793-D 5S,caminhao,OP_232,H_da819fcb7bee,335609935,rx channel b not receiving messages,3,informacional,0.0,null,0,false
139548941,2025-06-06T12:39:37.123Z,2025-06-06T06:00:00.000Z,2025-06-06T18:00:00.000Z,6,itabira,CA5926,793-D 5S,caminhao,OP_232,H_da819fcb7bee,335609934,rx channel a not receiving messages,3,informacional,0.0,null,0,false



--- Pré-visualização final: Regras de Alarmes ---


tipo,evento,situacao,qtd,tempo,nivel
alarme oem,low transmission oil level,mediante alarme nível 3,2,360,muito alto
alarme oem,low transmission oil level,mediante cinco alarmes nivel 2 consecutivos.,5,360,muito alto
alarme oem,transmission oil level - active,mediante alarme nível 3,2,360,muito alto
alarme oem,engine coolant level - active,mediante alarme nível 3,1,0,muito alto
alarme oem,engine coolant level - active,mediante dez alarmes nivel 2 consecutivos.,10,360,alto
alarme oem,low engine coolant level,mediante alarme nível 3,1,0,muito alto
alarme oem,low engine coolant level,mediante dez alarmes nivel 2 consecutivos.,10,360,alto
alarme oem,low coolant level,em qualquer situação,1,0,alto
alarme oem,engine coolant level low,mediante alarme nível 3,1,0,muito alto
alarme oem,engine coolant level low,em qualquer situação,1,0,alto


## 12. Controle de Alterações (ANTES/DEPOIS) — consolidado

In [0]:
df_controle_alteracoes = spark.createDataFrame(controle_alteracoes)
display(df_controle_alteracoes)

Campo,Justificativa,Problema_Identificado,Qtd_Registros,Tabela,Tratamento_Aplicado
_c0..c5 (cabeçalho embutido),"Conector de Excel na Bronze não reconheceu o header corretamente; tratado aqui para não perder a regra de negócio, mas recomendo corrigir a leitura na origem quando possível.",Linha de cabeçalho textual (TIPO/QTD/TEMPO) veio junto com os dados por falha de leitura na Bronze,1,df_regras_silver,Linha removida + colunas renomeadas para tipo/evento/situacao/qtd/tempo/nivel
"colunas de texto categóricas (Frota, Tipo, Classe, Alarme, Criticidade, Localidade, tipo/evento/situacao/nivel)","Evita que a mesma categoria seja contada como grupos distintos, sem correr o risco de corromper identificadores e hashes de anonimização, que são case-sensitive por natureza.",Capitalização e espaços em branco inconsistentes entre categorias,37164054,apontamentos / telemetria / regras,"trim() + lower() aplicado, exceto em Tag/TAG, hash do operador e colunas de Id"
valor,Zero pode ser uma leitura legítima do sensor para alguns tipos de alarme; imputar 0 cegamente confundiria 'sem leitura' com 'leitura zero'. A flag permite decidir a imputação por tipo de alarme na etapa de modelagem.,Valor nulo após conversão para double (não numérico ou ausente na origem),237443,df_telemetria_silver,Mantido nulo + criada a coluna flag 'valor_ausente' (não preenchido com 0.0)
"inicio, fim, duracao_minutos",Ciclo com duração nula/negativa não representa uma atividade real e distorceria qualquer feature de tempo de operação.,Inicio > Fim ou duração do ciclo <= 0,0,df_apontamentos_silver,Linha removida
flag_overlap_ciclo,"Sobreposição pode ser troca de operador/turno legítima ou erro de apontamento; decisão de remover exige análise caso a caso, não uma regra cega.",Ciclo com Inicio anterior ao Fim do ciclo anterior do mesmo equipamento,351,df_apontamentos_silver,Não removido — apenas sinalizado com flag_overlap_ciclo=True
duracao_minutos,Ciclos 'Parado' podem durar muitas horas legitimamente; remover cegamente distorceria a análise de equipamentos parados por manutenção real.,Duração de ciclo acima do limite IQR (> 141 min),0,df_apontamentos_silver,Não removido — sinalizado com flag_outlier_duracao=True
id,Id é chave única do ciclo; duplicidade indica reingestão do bronze.,Registros duplicados,0,df_apontamentos_silver,dropDuplicates(['id'])
id_eventos_telemetria,Chave única do evento; duplicidade vem de overlap entre arquivos mensais.,Registros duplicados,0,df_telemetria_silver,dropDuplicates(['id_eventos_telemetria'])
"tipo, evento, situacao",Mesma combinação de regra não deve aparecer duas vezes no catálogo.,Regras duplicadas,0,df_regras_silver,"dropDuplicates(['tipo','evento','situacao'])"


## 13. Gravação da camada Silver
Persisto as 4 tabelas como Delta no schema `vale_silver`.

In [0]:
(
    df_apontamentos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("`estudo-guiado`.vale_silver.apontamentos")
)

(
    df_telemetria_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("`estudo-guiado`.vale_silver.telemetria")
)

(
    df_regras_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("`estudo-guiado`.vale_silver.alarmes_regras")
)

(
    df_controle_alteracoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("`estudo-guiado`.vale_silver.controle_alteracoes")
)

print("====================================================================")
print(" Camada Silver gravada com sucesso!")
print(" -> estudo-guiado.vale_silver.apontamentos")
print(" -> estudo-guiado.vale_silver.telemetria")
print(" -> estudo-guiado.vale_silver.alarmes_regras")
print(" -> estudo-guiado.vale_silver.controle_alteracoes")
print("====================================================================")

 Camada Silver gravada com sucesso!
 -> estudo-guiado.vale_silver.apontamentos
 -> estudo-guiado.vale_silver.telemetria
 -> estudo-guiado.vale_silver.alarmes_regras
 -> estudo-guiado.vale_silver.controle_alteracoes
